In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench origin/main 7e5ec87 (pending 0.6.0).
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade21m_v5_ring")
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066784,0.067206,0.067469,0.067854,0.068547,0.069462,0.070648,0.072076,0.073268
Temperature (°C) [sea_water_potential_temperature]{surface},0.517910,0.519134,0.520646,0.524216,0.529996,0.538699,0.550304,0.562985,0.573220
Salinity (PSU) [sea_water_salinity]{surface},0.571852,0.567538,0.563205,0.559444,0.555637,0.552507,0.549891,0.547490,0.544489
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118931,0.119327,0.120124,0.121521,0.123407,0.125899,0.128885,0.132298,0.135036
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119967,0.120440,0.121335,0.122854,0.125065,0.127813,0.131119,0.134571,0.137200
Temperature (°C) [sea_water_potential_temperature]{50m},0.864551,0.863904,0.863062,0.862809,0.864600,0.867782,0.872367,0.878151,0.881443
Salinity (PSU) [sea_water_salinity]{50m},0.242370,0.242139,0.241823,0.241556,0.241457,0.241542,0.241763,0.242014,0.241956
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112329,0.112659,0.113044,0.113721,0.114716,0.116131,0.117958,0.120044,0.121440
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113354,0.113405,0.113672,0.114243,0.115101,0.116340,0.118114,0.120092,0.121430
Temperature (°C) [sea_water_potential_temperature]{100m},1.059568,1.059964,1.060710,1.062546,1.065398,1.070158,1.077033,1.083801,1.087373


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.135404,41.593482,41.98213,42.32308,42.748097,43.210571,43.748509,44.342161,44.723593


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.117067,0.117149,0.117545,0.118397,0.119849,0.121680,0.124407,0.127129,0.129200
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.123197,0.122847,0.123383,0.123948,0.125373,0.127285,0.130328,0.133228,0.136146


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Observations
Temperature (°C) [sea_water_potential_temperature]{surface},0.736203,0.760948,0.731049,0.756877,0.779478,0.820090,0.819130,0.836913,0.861318,150475
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.725385,0.739153,0.759745,0.790454,0.790988,0.800161,0.814675,0.811293,0.808105,80886
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.861993,0.872379,0.860843,0.894423,0.885092,0.886353,0.909301,0.937995,0.937050,1341567
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.781776,0.803830,0.795381,0.800887,0.817395,0.814926,0.841458,0.845853,0.876764,2132232
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.519994,0.533183,0.530414,0.533941,0.542469,0.560222,0.562896,0.575797,0.588094,2646322
Salinity (PSU) [sea_water_salinity]{0-5m},0.251071,0.277547,0.268152,0.304113,0.272641,0.279426,0.268551,0.262608,0.291217,69024
Salinity (PSU) [sea_water_salinity]{5-100m},0.195197,0.200347,0.204696,0.205665,0.205583,0.206029,0.208760,0.206749,0.210184,1143321
Salinity (PSU) [sea_water_salinity]{100-300m},0.126687,0.128682,0.129371,0.129535,0.132531,0.131087,0.137020,0.133560,0.136719,1816368
Salinity (PSU) [sea_water_salinity]{300-600m},0.080377,0.080779,0.080390,0.081153,0.082060,0.083969,0.083663,0.086882,0.087241,2245406
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048057,0.049027,0.049815,0.051333,0.052748,0.054596,0.056272,0.058004,0.060085,15465310


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},10.02873,19.377192,28.35601,37.071476,45.594563,53.958786,62.184612


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013845,0.016942,0.019796,0.023341,0.027504,0.031816,0.036281,0.040686,0.043829
Temperature (°C) [sea_water_potential_temperature]{surface},0.176395,0.207190,0.238057,0.269413,0.301811,0.335231,0.370622,0.404745,0.430030
Salinity (PSU) [sea_water_salinity]{surface},0.128429,0.148773,0.165699,0.180836,0.194844,0.210407,0.226110,0.240666,0.252428
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.051034,0.058938,0.066246,0.074240,0.082792,0.091291,0.100111,0.108406,0.114588
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.052546,0.060638,0.068165,0.076103,0.084574,0.093315,0.102280,0.110801,0.117065
Temperature (°C) [sea_water_potential_temperature]{50m},0.306777,0.336701,0.366837,0.399372,0.434624,0.473253,0.513371,0.549812,0.572730
Salinity (PSU) [sea_water_salinity]{50m},0.063420,0.069129,0.075407,0.082097,0.089273,0.096863,0.104654,0.111869,0.116762
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.045142,0.050611,0.056465,0.062710,0.069565,0.076780,0.084371,0.091493,0.096403
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.047120,0.052522,0.058160,0.064328,0.070980,0.078051,0.085457,0.092586,0.097650
Temperature (°C) [sea_water_potential_temperature]{100m},0.279462,0.320064,0.360730,0.404850,0.450899,0.498895,0.547997,0.592371,0.620828


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},31.141665,32.386596,33.391262,34.370775,35.40153,36.448491,37.478268,38.434736,39.185407


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.049486,0.058994,0.066742,0.074111,0.081465,0.088950,0.096861,0.104086,0.109317
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.052682,0.064206,0.073228,0.081083,0.088636,0.096819,0.104942,0.112391,0.117447


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},4.067334,8.094156,12.369177,17.048895,22.196806,27.831676,33.932884
